# MEFF v3 — full analysis of the ensemble (quantitative + significance + qualitative)

Runs on **CPU** (Accelerator = None is fine). No training. About 5–10 minutes.

**Inputs (Add Input):**
1. The dataset with the 5,955 images (the same one used for training).
2. One private dataset, e.g. `meff-v3-checkpoints`, containing nine files:
   - `best_s42.pth`, `best_s43.pth`, `best_s44.pth`. Download each from the **MEFF version** of that account's notebook: Output → `MEFF_v3/runs/meff_s4X/best.pth`. Rename it to `.pth` **before uploading**: Kaggle unzips `.zip` files, and that would break the checkpoint.
   - `meff_s42_results.zip`, `meff_s43_results.zip`, `meff_s44_results.zip` and `dsg_s42_results.zip`, `dsg_s43_results.zip`, `dsg_s44_results.zip` (the six result zips you already have).

The notebook identifies each checkpoint by its MD5 (recorded in `run_meta.json` at the end of training), so the file names do not matter.

**What it produces** (all in `MEFF_v3_analysis_outputs.zip`, figures as PNG 300 dpi + PDF):
- **Quantitative:** metrics of each model and the ensemble, 95% bootstrap confidence intervals for the ensemble, confusion matrices, ROC and precision-recall curves.
- **Significance tests:** the official pre-registered evaluation (`meff_v3.py eval`): paired bootstrap CI of Δ macro-F1, McNemar with Holm correction, and the decision.
- **Qualitative:** feature maps and Grad-CAM (each model and the ensemble; correct and misclassified test images).

Edit only the config cell, then **Run All**.

In [ ]:
# ===== EDIT ONLY THIS CELL =====
SEEDS = [42, 43, 44]
SEED_FOR_FEATUREMAPS = 42   # which model's feature maps to draw
N_CHECK_PER_CLASS = 8       # test images per class used to check each checkpoint against its saved predictions
# ================================
W = "/kaggle/working"; OUT = f"{W}/analysis"; M = f"{W}/MEFF_v3"
import os; os.makedirs(OUT, exist_ok=True); os.makedirs(M, exist_ok=True)

## 1. Model code — `meff_v3.py` (identical to training; do not edit)

In [ ]:
%%writefile /kaggle/working/MEFF_v3/meff_v3.py
#!/usr/bin/env python3
"""
MEFF v3 pipeline - one file, three commands.

  train       train ONE model (variant meff|dsg, one seed); resumable; stops cleanly before the wall limit
  eval        ensembles, metrics, paired group bootstrap, McNemar+Holm, decision rule (reads saved predictions)
  verify      check the dataset folder matches the frozen split (5,955 files, MD5s) before any training
  complexity  parameters, FLOPs and latency for one MEFF expert, the MEFF ensemble, and one DSG model

Variants
  meff : DenseNet-169 -> CCDGS -> Triplet Attention -> SE -> GAP -> [concat 42-D handcrafted] -> head
  dsg  : DenseNet-169 -> CCDGS -> Triplet Attention        -> GAP ->                          head
         (Dense-ShuffleGCANet re-implementation; Ajay, Bisht, Karthik, IEEE Access 13, 2025)

Examples
  python meff_v3.py train --variant meff --seed 42 --data_dir DFUC2021_deduplicated_5955 \
         --split_dir MEFF_v3/split --out_dir MEFF_v3/runs/meff_s42
  python meff_v3.py eval --runs_dir MEFF_v3/runs --split_dir MEFF_v3/split --out_dir MEFF_v3/results
  python meff_v3.py complexity --out_dir MEFF_v3/results
"""
import argparse, hashlib, json, os, platform, random, sys, time

import numpy as np
import pandas as pd

CLASSES = ["NONE", "INFECTION", "ISCHAEMIA", "BOTH"]
NC = len(CLASSES)

# ----------------------------------------------------------------------------------------------
# Frozen training recipe (pre-registered in PROTOCOL_v3.md - do not change after Part A)
# ----------------------------------------------------------------------------------------------
CFG = dict(
    img_size=224, batch_size=32, max_epochs=100, early_stop_patience=15,
    optimizer="Adam", lr=5e-5, weight_decay=1e-3,
    scheduler="ReduceLROnPlateau(mode=max on val macro-F1, factor=0.5, patience=5, min_lr=1e-7)",
    loss="FocalLoss(gamma=2.0, no alpha)", sampler="WeightedRandomSampler (inverse class frequency)",
    dropout=0.5, hidden=512, hc_dim=42, ccdgs_groups=4, se_reduction=16, ta_kernel=7,
    train_aug="RandomResizedCrop(224,0.8-1.0), HFlip, VFlip, Rotation(15), ColorJitter(0.2,0.2,0.2,0.02)",
    eval_aug="none (ToTensor + ImageNet normalisation)",
    checkpoint_selection="best validation macro-F1",
)


def md5_file(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def log(*a):
    print(time.strftime("[%Y-%m-%d %H:%M:%S]"), *a, flush=True)


# ----------------------------------------------------------------------------------------------
# Handcrafted 42-D descriptor: LBP (10) + RGB colour histogram (24) + edge-orientation histogram (8)
# ----------------------------------------------------------------------------------------------
def handcrafted_42(rgb):
    import cv2
    from skimage.feature import local_binary_pattern
    assert rgb.dtype == np.uint8 and rgb.ndim == 3 and rgb.shape[2] == 3
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    # LBP uniform P=8,R=1 -> codes 0..9 -> 10 bins
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_h = np.histogram(lbp, bins=np.arange(11))[0].astype(np.float64)
    lbp_h /= max(lbp_h.sum(), 1)
    # RGB colour histogram, 8 bins per channel
    col = []
    for ch in range(3):
        h = np.histogram(rgb[:, :, ch], bins=8, range=(0, 256))[0].astype(np.float64)
        col.append(h / max(h.sum(), 1))
    # Edge-orientation histogram: Sobel orientation at Canny edge pixels, 8 bins over [0, pi)
    eq = cv2.equalizeHist(cv2.GaussianBlur(gray, (5, 5), 0))  # blur + histogram equalisation (as in original MEFF)
    med = float(np.median(eq))
    edges = cv2.Canny(eq, int(max(0, 0.66 * med)), int(min(255, 1.33 * med)))
    gx = cv2.Sobel(eq, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(eq, cv2.CV_64F, 0, 1, ksize=3)
    ang = np.mod(np.arctan2(gy, gx), np.pi)[edges > 0]
    e_h = np.histogram(ang, bins=8, range=(0, np.pi))[0].astype(np.float64)
    e_h /= max(e_h.sum(), 1)
    f = np.concatenate([lbp_h, *col, e_h]).astype(np.float32)
    assert f.shape == (42,)
    return f


def load_features(data_dir, relpaths, cache_path):
    """Compute (or load cached) raw 42-D features for every relpath. Atomic write for parallel jobs."""
    from PIL import Image
    if os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=False)
        cached = dict(zip(z["relpath"].tolist(), z["feat"]))
        if all(r in cached for r in relpaths):
            return cached
    log(f"computing handcrafted features for {len(relpaths)} images ...")
    feats = np.stack([handcrafted_42(np.array(Image.open(os.path.join(data_dir, r)).convert("RGB")))
                      for r in relpaths])
    tmp = f"{cache_path}.{os.getpid()}.tmp.npz"
    np.savez(tmp, relpath=np.array(relpaths), feat=feats)
    os.replace(tmp, cache_path)
    return dict(zip(relpaths, feats))


# ----------------------------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------------------------
def build_model(variant, pretrained=True):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision

    class CCDGS(nn.Module):
        """Channel-Centric Depth-wise Group Shuffle block (spatial maps preserved, residual)."""
        def __init__(self, c, groups):
            super().__init__()
            self.g = groups
            self.gconv = nn.Conv2d(c, c, 3, padding=1, groups=groups, bias=False); self.bn1 = nn.BatchNorm2d(c)
            self.dw = nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False); self.bn2 = nn.BatchNorm2d(c)
            self.pw = nn.Conv2d(c, c, 1, bias=False); self.bn3 = nn.BatchNorm2d(c)

        def forward(self, x):
            y = F.relu(self.bn1(self.gconv(x)))
            b, c, h, w = y.shape
            y = y.view(b, self.g, c // self.g, h, w).transpose(1, 2).contiguous().view(b, c, h, w)  # shuffle
            y = F.relu(self.bn2(self.dw(y)))
            y = self.bn3(self.pw(y))
            return F.relu(x + y)

    class ZPool(nn.Module):
        def forward(self, x):
            return torch.cat([x.max(1, keepdim=True)[0], x.mean(1, keepdim=True)], 1)

    class Gate(nn.Module):
        def __init__(self, k):
            super().__init__()
            self.z = ZPool(); self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False); self.bn = nn.BatchNorm2d(1)

        def forward(self, x):
            return x * torch.sigmoid(self.bn(self.conv(self.z(x))))

    class TripletAttention(nn.Module):
        """Misra et al. (WACV 2021): C-H, C-W and H-W interaction branches, averaged."""
        def __init__(self, k):
            super().__init__()
            self.ch, self.cw, self.hw = Gate(k), Gate(k), Gate(k)

        def forward(self, x):
            o1 = self.ch(x.permute(0, 2, 1, 3).contiguous()).permute(0, 2, 1, 3)
            o2 = self.cw(x.permute(0, 3, 2, 1).contiguous()).permute(0, 3, 2, 1)
            o3 = self.hw(x)
            return (o1 + o2 + o3) / 3.0

    class SE(nn.Module):
        def __init__(self, c, r):
            super().__init__()
            self.fc = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True),
                                    nn.Linear(c // r, c, bias=False), nn.Sigmoid())

        def forward(self, x):
            return x * self.fc(x.mean((2, 3)))[:, :, None, None]

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            w = torchvision.models.DenseNet169_Weights.IMAGENET1K_V1 if pretrained else None
            self.features = torchvision.models.densenet169(weights=w).features
            self.ccdgs = CCDGS(1664, CFG["ccdgs_groups"])
            self.ta = TripletAttention(CFG["ta_kernel"])
            self.se = SE(1664, CFG["se_reduction"]) if variant == "meff" else nn.Identity()
            self.use_hc = variant == "meff"
            fin = 1664 + (CFG["hc_dim"] if self.use_hc else 0)
            self.head = nn.Sequential(nn.Dropout(CFG["dropout"]), nn.Linear(fin, CFG["hidden"]), nn.ReLU(inplace=True),
                                      nn.Dropout(CFG["dropout"]), nn.Linear(CFG["hidden"], NC))

        def forward(self, x, hc=None):
            f = F.relu(self.features(x))
            f = self.se(self.ta(self.ccdgs(f)))
            f = f.mean((2, 3))
            if self.use_hc:
                assert hc is not None and hc.shape[1] == CFG["hc_dim"]
                f = torch.cat([f, hc], 1)
            return self.head(f)

    assert variant in ("meff", "dsg")
    return Net()


# ----------------------------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------------------------
def read_split(split_dir, limit_per_class=None):
    sp = {}
    for n in ["train", "validation", "test"]:
        d = pd.read_csv(os.path.join(split_dir, f"{n}.csv"))
        if limit_per_class:
            d = d.groupby("label", group_keys=False).head(limit_per_class).reset_index(drop=True)
        sp[n] = d
    for k in ["relpath", "md5", "group_id"]:
        s = {n: set(d[k]) for n, d in sp.items()}
        assert not (s["train"] & s["validation"]) and not (s["train"] & s["test"]) and not (s["validation"] & s["test"]), \
            f"leakage: {k} shared between subsets"
    return sp


def make_loaders(sp, data_dir, feats, seed, batch, workers, img_size):
    import torch
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    import torchvision.transforms as T
    from PIL import Image

    norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    train_tf = T.Compose([T.RandomResizedCrop(img_size, scale=(0.8, 1.0)), T.RandomHorizontalFlip(),
                          T.RandomVerticalFlip(), T.RandomRotation(15),
                          T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), norm])
    eval_tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), norm])

    tr = np.stack([feats[r] for r in sp["train"].relpath])
    mu, sd = tr.mean(0), tr.std(0)
    sd[sd < 1e-6] = 1.0  # standardisation fitted on TRAIN only

    class DS(Dataset):
        def __init__(self, d, tf):
            self.p = d.relpath.tolist(); self.y = d.label.astype(int).tolist(); self.tf = tf
            self.h = [((feats[r] - mu) / sd).astype(np.float32) for r in self.p]

        def __len__(self):
            return len(self.p)

        def __getitem__(self, i):
            # transform is chosen by SPLIT only, never by label
            img = Image.open(os.path.join(data_dir, self.p[i])).convert("RGB")
            return self.tf(img), self.y[i], torch.from_numpy(self.h[i])

    y = sp["train"].label.values
    cw = 1.0 / np.bincount(y, minlength=NC)
    g = torch.Generator(); g.manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(cw[y], dtype=torch.double), len(y), replacement=True, generator=g)

    def wif(wid):
        s = torch.initial_seed() % 2 ** 32
        np.random.seed(s); random.seed(s)

    kw = dict(batch_size=batch, num_workers=workers, worker_init_fn=wif, pin_memory=torch.cuda.is_available())
    dl = dict(train=DataLoader(DS(sp["train"], train_tf), sampler=sampler, **kw),
              validation=DataLoader(DS(sp["validation"], eval_tf), shuffle=False, **kw),
              test=DataLoader(DS(sp["test"], eval_tf), shuffle=False, **kw))
    return dl, g, (train_tf, eval_tf)


def assert_eval_transform_ok(eval_tf):
    import torchvision.transforms as T
    rnd = (T.RandomResizedCrop, T.RandomHorizontalFlip, T.RandomVerticalFlip, T.RandomRotation, T.ColorJitter,
           T.RandomAffine, T.RandomCrop, T.RandomErasing, T.RandomApply, T.RandomChoice)
    assert not any(isinstance(t, rnd) for t in eval_tf.transforms), "eval transform contains random ops"


# ----------------------------------------------------------------------------------------------
# Metrics (fast numpy versions; used in training and bootstrap)
# ----------------------------------------------------------------------------------------------
def confusion(y, p):
    return np.bincount(y * NC + p, minlength=NC * NC).reshape(NC, NC)


def macro_f1(y, p):
    cm = confusion(y, p).astype(float)
    tp = np.diag(cm); prec_d = cm.sum(0); rec_d = cm.sum(1)
    f1 = np.where(prec_d + rec_d > 0, 2 * tp / np.maximum(prec_d + rec_d, 1e-12), 0.0)
    return float(f1.mean())


# ----------------------------------------------------------------------------------------------
# TRAIN
# ----------------------------------------------------------------------------------------------
def cmd_train(a):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    t_start = time.time()
    os.makedirs(a.out_dir, exist_ok=True)
    done_flag = os.path.join(a.out_dir, "DONE")
    if os.path.exists(done_flag):
        log(f"{a.out_dir} already DONE - nothing to do"); return 0

    smoke = a.smoke
    if smoke:
        CFG.update(max_epochs=a.smoke_epochs, early_stop_patience=99, batch_size=a.smoke_batch)
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = dev.type == "cuda"

    sp = read_split(a.split_dir, limit_per_class=16 if smoke else None)
    allrel = sorted(pd.concat(sp.values()).relpath)
    cache = os.path.join(os.path.dirname(os.path.abspath(a.split_dir)), "features42_cache.npz")  # next to split/, not inside
    feats = load_features(a.data_dir, allrel, cache if not smoke else os.path.join(a.out_dir, "features42_smoke.npz"))
    dl, gen, (train_tf, eval_tf) = make_loaders(sp, a.data_dir, feats, a.seed, CFG["batch_size"], a.workers, CFG["img_size"])
    assert_eval_transform_ok(eval_tf)

    model = build_model(a.variant, pretrained=not a.no_pretrained).to(dev)
    if not a.no_pretrained:
        # ImageNet weights check: first conv must equal the torchvision ImageNet tensor
        import torchvision
        ref = torchvision.models.densenet169(weights=torchvision.models.DenseNet169_Weights.IMAGENET1K_V1).features.conv0.weight
        assert torch.allclose(model.features.conv0.weight.detach().cpu(), ref), "ImageNet weights not loaded"
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5, min_lr=1e-7)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def focal(logits, y, gamma=2.0):
        ce = F.cross_entropy(logits.float(), y, reduction="none")
        return ((1 - torch.exp(-ce)) ** gamma * ce).mean()

    last_p, best_p = os.path.join(a.out_dir, "last.pth"), os.path.join(a.out_dir, "best.pth")
    st = dict(epoch=0, best_f1=-1.0, best_epoch=-1, bad=0, history=[])
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); st = ck["state"]; gen.set_state(ck["gen"])
        torch.set_rng_state(ck["rng_torch"]); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
        if use_amp and ck.get("rng_cuda") is not None:
            torch.cuda.set_rng_state_all(ck["rng_cuda"])
        log(f"RESUMED from epoch {st['epoch']} (best macro-F1 {st['best_f1']:.4f} @ epoch {st['best_epoch']})")

    meta = dict(variant=a.variant, seed=a.seed, smoke=smoke, pretrained=not a.no_pretrained, cfg=CFG,
                split_md5={n: md5_file(os.path.join(a.split_dir, f"{n}.csv")) for n in ["train", "validation", "test"]},
                script_md5=md5_file(os.path.abspath(__file__)), python=platform.python_version(),
                torch=torch.__version__, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                n=dict((k, len(v)) for k, v in sp.items()),
                params=sum(p.numel() for p in model.parameters()))
    json.dump(meta, open(os.path.join(a.out_dir, "run_meta.json"), "w"), indent=2)
    log(f"variant={a.variant} seed={a.seed} device={meta['gpu']} params={meta['params']:,} n={meta['n']}")

    def run_eval(loader):
        model.eval(); P, Y, L = [], [], 0.0
        with torch.no_grad():
            for x, y, h in loader:
                x, y, h = x.to(dev, non_blocking=True), y.to(dev), h.to(dev)
                with torch.autocast(device_type=dev.type, enabled=use_amp):
                    out = model(x, h)
                L += focal(out, y).item() * len(y)
                P.append(torch.softmax(out.float(), 1).cpu().numpy()); Y.append(y.cpu().numpy())
        P, Y = np.concatenate(P), np.concatenate(Y)
        return P, Y, L / len(Y)

    epoch_times = []
    while st["epoch"] < CFG["max_epochs"] and st["bad"] < CFG["early_stop_patience"]:
        t0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y, h in dl["train"]:
            x, y, h = x.to(dev, non_blocking=True), y.to(dev), h.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                out = model(x, h)
            loss = focal(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        Pv, Yv, vl = run_eval(dl["validation"])
        vf1 = macro_f1(Yv, Pv.argmax(1)); vacc = float((Pv.argmax(1) == Yv).mean())
        sched.step(vf1)
        st["epoch"] += 1
        improved = vf1 > st["best_f1"]
        if improved:
            st.update(best_f1=vf1, best_epoch=st["epoch"], bad=0)
            torch.save(dict(model=model.state_dict(), epoch=st["epoch"], val_macro_f1=vf1), best_p)
        else:
            st["bad"] += 1
        st["history"].append(dict(epoch=st["epoch"], train_loss=tl / tn, train_acc=tc / tn, val_loss=vl,
                                  val_acc=vacc, val_macro_f1=vf1, lr=opt.param_groups[0]["lr"]))
        pd.DataFrame(st["history"]).to_csv(os.path.join(a.out_dir, "history.csv"), index=False)
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), state=st, gen=gen.get_state(),
                        rng_torch=torch.get_rng_state(), rng_np=np.random.get_state(), rng_py=random.getstate(),
                        rng_cuda=torch.cuda.get_rng_state_all() if use_amp else None), last_p + ".tmp")
        os.replace(last_p + ".tmp", last_p)
        epoch_times.append(time.time() - t0)
        log(f"ep {st['epoch']:3d} | train loss {tl/tn:.4f} acc {tc/tn:.4f} | val loss {vl:.4f} acc {vacc:.4f} "
            f"macroF1 {vf1:.4f}{' *best*' if improved else ''} | lr {opt.param_groups[0]['lr']:.1e} | {epoch_times[-1]:.0f}s"
            + (f" | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if use_amp else ""))
        elapsed_h = (time.time() - t_start) / 3600
        if elapsed_h + 1.5 * max(epoch_times) / 3600 > a.max_hours:
            log(f"time guard: {elapsed_h:.2f} h used of {a.max_hours} h - state saved, exiting for resubmission")
            return 0

    log(f"training finished: epochs {st['epoch']}, best val macro-F1 {st['best_f1']:.4f} @ epoch {st['best_epoch']}")
    ck = torch.load(best_p, map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"])
    for n in ["validation", "test"]:
        P, Y, _ = run_eval(dl[n])
        d = sp[n][["relpath", "group_id"]].copy()
        d["true"] = Y; d["pred"] = P.argmax(1)
        for i, c in enumerate(CLASSES):
            d[f"prob_{c}"] = P[:, i]
        d.to_csv(os.path.join(a.out_dir, f"{'val' if n == 'validation' else n}_predictions.csv"), index=False)
        log(f"{n}: acc {(d.true == d.pred).mean():.4f} macroF1 {macro_f1(d.true.values, d.pred.values):.4f} "
            f"(best checkpoint, epoch {ck['epoch']})")
    meta.update(best_epoch=st["best_epoch"], best_val_macro_f1=st["best_f1"], epochs_run=st["epoch"],
                best_ckpt_md5=md5_file(best_p))
    json.dump(meta, open(os.path.join(a.out_dir, "run_meta.json"), "w"), indent=2)
    open(done_flag, "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
    if os.path.exists(last_p) and not a.keep_last:
        os.remove(last_p)
    return 0


# ----------------------------------------------------------------------------------------------
# EVAL
# ----------------------------------------------------------------------------------------------
def full_metrics(y, pred, prob):
    from sklearn.metrics import roc_auc_score, f1_score
    cm = confusion(y, pred)
    r = dict(n=len(y), accuracy=float((y == pred).mean()), macro_f1=macro_f1(y, pred),
             weighted_f1=float(f1_score(y, pred, average="weighted", labels=list(range(NC)), zero_division=0)))
    try:
        r["macro_auc"] = float(roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=list(range(NC))))
    except ValueError:
        r["macro_auc"] = float("nan")
    for i, c in enumerate(CLASSES):
        tp, fp, fn = cm[i, i], cm[:, i].sum() - cm[i, i], cm[i, :].sum() - cm[i, i]
        p = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        r[f"{c}_precision"] = float(p); r[f"{c}_recall"] = float(rc)
        r[f"{c}_f1"] = float(2 * p * rc / (p + rc)) if p + rc else 0.0
        r[f"{c}_correct"] = int(tp); r[f"{c}_total"] = int(cm[i, :].sum())
    return r


def hard_vote(preds, probs):
    """Majority vote; 3-way tie -> class with highest mean softmax probability."""
    preds = np.stack(preds, 1); mp = np.mean(probs, 0)
    out, ties = np.empty(len(preds), int), 0
    for i, row in enumerate(preds):
        cnt = np.bincount(row, minlength=NC)
        top = np.flatnonzero(cnt == cnt.max())
        if len(top) > 1:
            ties += 1
            out[i] = top[np.argmax(mp[i, top])]
        else:
            out[i] = top[0]
    return out, mp, ties


def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i])); adj[i] = run
    return adj


def cmd_eval(a):
    from scipy.stats import binomtest
    os.makedirs(a.out_dir, exist_ok=True)
    seeds = a.seeds
    runs, ref = {}, None
    for v in ["meff", "dsg"]:
        for s in seeds:
            rd = os.path.join(a.runs_dir, f"{v}_s{s}")
            if not os.path.exists(os.path.join(rd, "DONE")):
                sys.exit(f"ERROR: {rd} is not DONE - finish or resume that run first")
            meta = json.load(open(os.path.join(rd, "run_meta.json")))
            if (meta["smoke"] or not meta["pretrained"]) and not a.allow_smoke:
                sys.exit(f"ERROR: {rd} is a smoke/no-pretrained run - not valid for results")
            te = pd.read_csv(os.path.join(rd, "test_predictions.csv"))
            va = pd.read_csv(os.path.join(rd, "val_predictions.csv"))
            if ref is None:
                ref = te[["relpath", "group_id", "true"]]
            assert te.relpath.equals(ref.relpath) and te.true.equals(ref.true), f"{rd}: test rows differ"
            runs[(v, s)] = dict(te=te, va=va, meta=meta)
    y = ref.true.values; groups = ref.group_id.values
    pc = [f"prob_{c}" for c in CLASSES]
    rows, preds = [], {}
    for v in ["meff", "dsg"]:
        for s in seeds:
            te = runs[(v, s)]["te"]; nm = f"{v.upper()}_s{s}"
            preds[nm] = te.pred.values
            rows.append(dict(model=nm, **full_metrics(y, te.pred.values, te[pc].values)))
        seed_rows = pd.DataFrame([r for r in rows if r["model"].startswith(v.upper() + "_s")]).drop(columns="model")
        rows.append(dict(model=f"{v.upper()}_seed_mean", **seed_rows.mean().to_dict()))
        rows.append(dict(model=f"{v.upper()}_seed_sd", **seed_rows.std(ddof=1).to_dict()))
    ties = {}
    for v in ["meff", "dsg"]:
        ps = [runs[(v, s)]["te"].pred.values for s in seeds]
        pr = [runs[(v, s)]["te"][pc].values for s in seeds]
        hv, mp, nt = hard_vote(ps, pr); ties[v.upper()] = nt
        preds[f"{v.upper()}_hard"] = hv; preds[f"{v.upper()}_soft"] = mp.argmax(1)
        rows.append(dict(model=f"{v.upper()}_hard", **full_metrics(y, hv, mp)))
        rows.append(dict(model=f"{v.upper()}_soft", **full_metrics(y, mp.argmax(1), mp)))
        pd.DataFrame(confusion(y, hv), index=[f"true_{c}" for c in CLASSES],
                     columns=[f"pred_{c}" for c in CLASSES]).to_csv(os.path.join(a.out_dir, f"confusion_{v.upper()}_hard.csv"))
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(a.out_dir, "results.csv"), index=False)
    pd.DataFrame([dict(ensemble=k, three_way_ties=v, n=len(y)) for k, v in ties.items()]).to_csv(
        os.path.join(a.out_dir, "ensemble_ties.csv"), index=False)

    # best single MEFF seed chosen on VALIDATION macro-F1 (never on test)
    vf = {s: macro_f1(runs[("meff", s)]["va"].true.values, runs[("meff", s)]["va"].pred.values) for s in seeds}
    best_s = max(seeds, key=lambda s: vf[s])
    comps = [("MEFF_hard", "DSG_hard")] + [("MEFF_hard", f"DSG_s{s}") for s in seeds] + [("MEFF_hard", f"MEFF_s{best_s}")]

    # paired group-level bootstrap
    ug = np.unique(groups); idx_by_g = {g: np.flatnonzero(groups == g) for g in ug}
    rng = np.random.default_rng(42)
    samples = [np.concatenate([idx_by_g[g] for g in rng.choice(ug, len(ug), replace=True)]) for _ in range(a.n_boot)]
    out = []
    for A, B in comps:
        pa, pb = preds[A], preds[B]
        df1 = [macro_f1(y[i], pa[i]) - macro_f1(y[i], pb[i]) for i in samples]
        dac = [(y[i] == pa[i]).mean() - (y[i] == pb[i]).mean() for i in samples]
        ca, cb = pa == y, pb == y
        b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
        p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
        out.append(dict(A=A, B=B, delta_macro_f1=macro_f1(y, pa) - macro_f1(y, pb),
                        ci95_low_macro_f1=float(np.percentile(df1, 2.5)), ci95_high_macro_f1=float(np.percentile(df1, 97.5)),
                        delta_accuracy=float(ca.mean() - cb.mean()),
                        ci95_low_accuracy=float(np.percentile(dac, 2.5)), ci95_high_accuracy=float(np.percentile(dac, 97.5)),
                        mcnemar_b_A_right_B_wrong=b, mcnemar_c_A_wrong_B_right=c, mcnemar_p_exact=float(p)))
    comp = pd.DataFrame(out); comp["mcnemar_p_holm"] = holm(comp.mcnemar_p_exact.values)
    comp["best_meff_seed_by_val_macro_f1"] = best_s
    comp.to_csv(os.path.join(a.out_dir, "comparison.csv"), index=False)

    prim = comp.iloc[0]
    if prim.ci95_low_macro_f1 > 0:
        verdict = "MEFF outperforms Dense-ShuffleGCANet"
    elif prim.ci95_high_macro_f1 < 0:
        verdict = "Dense-ShuffleGCANet outperforms MEFF"
    else:
        verdict = "No significant difference between MEFF and Dense-ShuffleGCANet"
    txt = (f"Primary comparison: MEFF_hard - DSG_hard, test macro-F1\n"
           f"delta = {prim.delta_macro_f1:.6f}, 95% CI [{prim.ci95_low_macro_f1:.6f}, {prim.ci95_high_macro_f1:.6f}] "
           f"(paired group bootstrap, {a.n_boot} resamples, seed 42, {len(ug)} test groups)\n"
           f"Rule: CI entirely > 0 -> MEFF outperforms; entirely < 0 -> DSG outperforms; else no significant difference.\n"
           f"DECISION: {verdict}\n")
    open(os.path.join(a.out_dir, "decision.txt"), "w").write(txt)
    prot = os.path.join(os.path.dirname(os.path.abspath(a.split_dir)), "PROTOCOL_v3.md")
    rec = dict(protocol_md5=md5_file(prot) if os.path.exists(prot) else None, script_md5=md5_file(os.path.abspath(__file__)),
               runs={f"{v}_s{s}": runs[(v, s)]["meta"].get("best_ckpt_md5") for v in ["meff", "dsg"] for s in seeds})
    json.dump(rec, open(os.path.join(a.out_dir, "eval_meta.json"), "w"), indent=2)
    pd.set_option("display.width", 200)
    print(res[["model", "n", "accuracy", "macro_f1", "weighted_f1", "macro_auc", "ISCHAEMIA_recall", "BOTH_recall"]].round(4).to_string(index=False))
    print(comp[["A", "B", "delta_macro_f1", "ci95_low_macro_f1", "ci95_high_macro_f1", "mcnemar_p_holm"]].round(4).to_string(index=False))
    print(txt)
    return 0


# ----------------------------------------------------------------------------------------------
# VERIFY: dataset on disk must match the frozen split (every file present, MD5 identical, nothing extra)
# ----------------------------------------------------------------------------------------------
def cmd_verify(a):
    sp = read_split(a.split_dir)
    allr = pd.concat(sp.values())
    on_disk = {os.path.relpath(os.path.join(r, f), a.data_dir) for r, _, fs in os.walk(a.data_dir)
               for f in fs if f.lower().endswith((".jpg", ".jpeg", ".png"))}
    missing = [r for r in allr.relpath if r not in on_disk]
    extra = sorted(on_disk - set(allr.relpath))
    bad = [r for r, m in zip(allr.relpath, allr.md5) if r in on_disk and md5_file(os.path.join(a.data_dir, r)) != m]
    print(f"split rows {len(allr)} | files on disk {len(on_disk)} | missing {len(missing)} | extra {len(extra)} | md5 mismatch {len(bad)}")
    for n, d in sp.items():
        print(f"  {n:10s} {len(d):5d}  " + "  ".join(f"{c}={int((d.class_name == c).sum())}" for c in CLASSES))
    man = os.path.join(a.split_dir, "SPLIT_MANIFEST.json")
    if os.path.exists(man):
        exp = json.load(open(man))["files_md5"]
        for f, m in exp.items():
            ok = md5_file(os.path.join(a.split_dir, f)) == m
            print(f"  {f:16s} md5 {'OK' if ok else 'CHANGED'}")
            bad += [] if ok else [f]
    if missing or extra or bad or len(allr) != 5955:
        print("VERIFY FAILED - do not train"); return 1
    print("VERIFY OK"); return 0


# ----------------------------------------------------------------------------------------------
# COMPLEXITY
# ----------------------------------------------------------------------------------------------
def cmd_complexity(a):
    import torch
    from torch.utils.flop_counter import FlopCounterMode
    os.makedirs(a.out_dir, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []
    for v in ["meff", "dsg"]:
        m = build_model(v, pretrained=False).to(dev).eval()
        x = torch.randn(1, 3, 224, 224, device=dev); h = torch.randn(1, 42, device=dev)
        with torch.no_grad():
            fc = FlopCounterMode(display=False)
            with fc:
                m(x, h)
            flops = fc.get_total_flops()
            for _ in range(5):
                m(x, h)
            if dev.type == "cuda":
                torch.cuda.synchronize()
            ts = []
            for _ in range(a.n_timing):
                t0 = time.perf_counter(); m(x, h)
                if dev.type == "cuda":
                    torch.cuda.synchronize()
                ts.append(time.perf_counter() - t0)
        prm = sum(p.numel() for p in m.parameters())
        rows.append(dict(model="MEFF expert" if v == "meff" else "Dense-ShuffleGCANet (DSG)", params=prm,
                         params_M=prm / 1e6, gflops=flops / 1e9, gmacs=flops / 2e9,
                         latency_ms_batch1_median=1000 * float(np.median(ts)), device=str(dev)))
        if v == "meff":
            rows.append(dict(model="MEFF ensemble (3 experts)", params=3 * prm, params_M=3 * prm / 1e6,
                             gflops=3 * flops / 1e9, gmacs=3 * flops / 2e9,
                             latency_ms_batch1_median=3000 * float(np.median(ts)), device=str(dev)))
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(a.out_dir, "complexity_v3.csv"), index=False)
    print(df.round(3).to_string(index=False))
    return 0


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train")
    t.add_argument("--variant", choices=["meff", "dsg"], required=True)
    t.add_argument("--seed", type=int, required=True)
    t.add_argument("--data_dir", required=True)
    t.add_argument("--split_dir", required=True)
    t.add_argument("--out_dir", required=True)
    t.add_argument("--workers", type=int, default=4)
    t.add_argument("--max_hours", type=float, default=46.0, help="stop cleanly before this many hours")
    t.add_argument("--keep_last", action="store_true", help="keep last.pth after DONE")
    t.add_argument("--no_pretrained", action="store_true", help="SMOKE TESTS ONLY")
    t.add_argument("--smoke", action="store_true", help="16 images/class/split, few epochs - pipeline test only")
    t.add_argument("--smoke_epochs", type=int, default=2)
    t.add_argument("--smoke_batch", type=int, default=4, help="batch size in smoke mode only (real runs: 32)")
    e = sub.add_parser("eval")
    e.add_argument("--runs_dir", required=True)
    e.add_argument("--split_dir", required=True)
    e.add_argument("--out_dir", required=True)
    e.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44])
    e.add_argument("--n_boot", type=int, default=2000)
    e.add_argument("--allow_smoke", action="store_true")
    v = sub.add_parser("verify")
    v.add_argument("--data_dir", required=True)
    v.add_argument("--split_dir", required=True)
    c = sub.add_parser("complexity")
    c.add_argument("--out_dir", required=True)
    c.add_argument("--n_timing", type=int, default=50)
    a = ap.parse_args()
    return dict(train=cmd_train, eval=cmd_eval, verify=cmd_verify, complexity=cmd_complexity)[a.cmd](a)


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import hashlib, importlib.util
assert hashlib.md5(open(f"{M}/meff_v3.py", "rb").read()).hexdigest() == "d4a5febc73c33620713a457530f100b4", "meff_v3.py was changed"
spec = importlib.util.spec_from_file_location("meff_v3", f"{M}/meff_v3.py"); mv = importlib.util.module_from_spec(spec); spec.loader.exec_module(mv)
C, NC = mv.CLASSES, mv.NC
print("meff_v3.py OK")

## 2. Find the images, the result files and the checkpoints

In [ ]:
import glob, json, zipfile
import numpy as np, pandas as pd

# images: the folder that holds NONE/INFECTION/ISCHAEMIA/BOTH
cands = set()
for depth in range(1, 6):
    for p in glob.glob("/kaggle/input/" + "*/" * depth + "ISCHAEMIA"):
        d = os.path.dirname(p)
        if all(os.path.isdir(os.path.join(d, k)) for k in C): cands.add(d)
    if cands: break
assert len(cands) == 1, f"image folder not found exactly once: {cands}"
DATA = cands.pop(); print("images:", DATA)

# result files: use extracted folders if Kaggle unzipped the zips, otherwise extract them here
def find_run(s, v="meff"):   # skip smoke-test runs
    hits = [os.path.dirname(h) for h in glob.glob(f"/kaggle/input/**/{v}_s{s}/run_meta.json", recursive=True)
            if not json.load(open(h))["smoke"] and os.path.exists(os.path.join(os.path.dirname(h), "DONE"))]
    if hits: return hits[0]
    zips = [z for z in glob.glob(f"/kaggle/input/**/{v}_s{s}_results*.zip", recursive=True) if "SMOKE" not in z]
    assert zips, f"{v}_s{s}_results.zip not found in the inputs"
    zipfile.ZipFile(zips[0]).extractall(f"{W}/results"); return f"{W}/results/{v}_s{s}"
RUN = {s: find_run(s) for s in SEEDS}
# DSG result zips are OPTIONAL: they are only used by section 3b (the pre-registered MEFF-vs-DSG test)
try:
    RUN_DSG = {s: find_run(s, "dsg") for s in SEEDS}; HAVE_DSG = True
except AssertionError:
    RUN_DSG = {}; HAVE_DSG = False
if HAVE_DSG:
    for s in SEEDS:
        md_ = json.load(open(f"{RUN_DSG[s]}/run_meta.json"))
        assert md_["variant"] == "dsg" and md_["seed"] == s, f"dsg_s{s}: the zip is not the DSG seed-{s} run (variant={md_['variant']}, seed={md_['seed']})"
    import shutil   # all six runs side by side, as the official evaluation expects them
    for v, D in [("meff", RUN), ("dsg", RUN_DSG)]:
        for s in SEEDS:
            shutil.copytree(D[s], f"{W}/runs/{v}_s{s}", dirs_exist_ok=True)
print("DSG results found - section 3b will run" if HAVE_DSG else "No DSG results - section 3b (MEFF vs DSG) is skipped; everything else uses MEFF only")
meta = {s: json.load(open(f"{RUN[s]}/run_meta.json")) for s in SEEDS}
te = {s: pd.read_csv(f"{RUN[s]}/test_predictions.csv") for s in SEEDS}
for s in SEEDS:
    assert meta[s]["variant"] == "meff" and not meta[s]["smoke"] and meta[s]["pretrained"], f"seed {s}: not a valid MEFF run"
    assert meta[s]["script_md5"] == "d4a5febc73c33620713a457530f100b4", f"seed {s}: different code"
    assert te[s].relpath.equals(te[SEEDS[0]].relpath), "test rows differ between runs"
SPLIT = f"{RUN[SEEDS[0]]}/split"
print({s: RUN[s] for s in SEEDS})

# checkpoints: match every .pth in the inputs to a run by MD5
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
pths = {md5(p): p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)}
CKPT = {}
for s in SEEDS:
    want = meta[s]["best_ckpt_md5"]
    assert want in pths, (f"seed {s}: no .pth with MD5 {want}. Download best.pth from the MEFF version of the seed-{s} notebook, "
                          "rename it to .pth (not .zip) and upload it. Found: " + str(list(pths.values())))
    CKPT[s] = pths[want]
if glob.glob("/kaggle/input/**/data.pkl", recursive=True):
    print("NOTE: an unzipped checkpoint folder (data.pkl) was found - that upload was a .zip; it is ignored.")
for s in SEEDS: print(f"seed {s}: {CKPT[s]}  (MD5 matches run_meta.json)")

## 3. MEFF ensemble metrics on the 1,191 test images (from the saved predictions)

In [ ]:
PC = [f"prob_{k}" for k in C]
y = te[SEEDS[0]].true.values
hv, mp, ties = mv.hard_vote([te[s].pred.values for s in SEEDS], [te[s][PC].values for s in SEEDS])
rows = [dict(model=f"Expert {SEEDS.index(s)+1} (seed {s})", **mv.full_metrics(y, te[s].pred.values, te[s][PC].values)) for s in SEEDS]
rows += [dict(model="MEFF - hard vote (primary)", **mv.full_metrics(y, hv, mp)),
         dict(model="MEFF - soft vote (secondary)", **mv.full_metrics(y, mp.argmax(1), mp))]
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/meff_ensemble_metrics.csv", index=False)
cols = ["model", "accuracy", "macro_f1", "weighted_f1", "macro_auc"] + [f"{k}_recall" for k in C]
pd.set_option("display.width", 220)
print((res[cols].set_index("model") * 100).round(2).to_string())
print(f"hard-vote ties: {ties} of {len(y)}")
cm = pd.DataFrame(mv.confusion(y, hv), index=[f"true {k}" for k in C], columns=[f"pred {k}" for k in C])
cm.to_csv(f"{OUT}/confusion_MEFF_hard.csv"); print(cm)

## 3a. Significance tests within MEFF — ensemble vs each single MEFF model
Uses only the three MEFF runs. Paired bootstrap 95% CI of Δ macro-F1 (2,000 resamples, seed 42) and McNemar exact test,
Holm-corrected over the three comparisons. **Exploratory**, except the comparison with the best single model on validation
(seed 44), which is also part of the pre-registered evaluation in 3b.

In [ ]:
from scipy.stats import binomtest
rng = np.random.default_rng(42); idx = [rng.integers(0, len(y), len(y)) for _ in range(2000)]
rows_ = []
for s in SEEDS:
    p1 = te[s].pred.values; d = [mv.macro_f1(y[i], hv[i]) - mv.macro_f1(y[i], p1[i]) for i in idx]
    a_, b_ = int(((hv == y) & (p1 != y)).sum()), int(((hv != y) & (p1 == y)).sum())
    rows_.append(dict(comparison=f"MEFF - Expert {SEEDS.index(s)+1} (seed {s})", delta_macro_f1=mv.macro_f1(y, hv) - mv.macro_f1(y, p1),
                      ci95_low=np.percentile(d, 2.5), ci95_high=np.percentile(d, 97.5),
                      ensemble_right_single_wrong=a_, ensemble_wrong_single_right=b_,
                      mcnemar_p=binomtest(min(a_, b_), a_ + b_, 0.5).pvalue if a_ + b_ else 1.0))
sig = pd.DataFrame(rows_); sig["mcnemar_p_holm"] = mv.holm(sig.mcnemar_p.values)
sig["CI excludes 0"] = (sig.ci95_low > 0) | (sig.ci95_high < 0)
sig.to_csv(f"{OUT}/meff_ensemble_vs_single_models.csv", index=False)
print(sig.round(4).to_string(index=False))

## 3b. MEFF vs Dense-ShuffleGCANet — the official pre-registered evaluation (needs the 3 DSG zips; skipped otherwise)
Runs `meff_v3.py eval` on all six runs, exactly as fixed before training:
- primary comparison: MEFF ensemble vs DSG ensemble (hard vote), test macro-F1;
- 95% CI of the difference from 2,000 paired bootstrap resamples (seed 42);
- McNemar exact test with Holm correction over the five pre-registered comparisons;
- decision rule: CI fully above 0 → MEFF better; fully below 0 → DSG better; otherwise no significant difference.

In [ ]:
import subprocess, sys
if not HAVE_DSG:
    print("Section 3b skipped: attach the three dsg_s4X_results.zip files to run the pre-registered MEFF-vs-DSG test.")
else:
  r = subprocess.run([sys.executable, f"{M}/meff_v3.py", "eval", "--runs_dir", f"{W}/runs", "--split_dir", SPLIT,
                    "--out_dir", f"{OUT}/official_eval"], capture_output=True, text=True)
  print(r.stdout[-6000:], r.stderr[-1500:]); assert r.returncode == 0, "evaluation failed - see above"
  print(open(f"{OUT}/official_eval/decision.txt").read())

## 3c. 95% confidence intervals for the MEFF ensemble's metrics
Image-level bootstrap of the 1,191 test images (2,000 resamples, seed 42, percentile interval). Descriptive only.

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score
rng = np.random.default_rng(42); B = 2000; n = len(y); boot = []
for _ in range(B):
    i = rng.integers(0, n, n); yi, pi, qi = y[i], hv[i], mp[i]
    row = dict(accuracy=(yi == pi).mean(), macro_f1=mv.macro_f1(yi, pi),
               weighted_f1=f1_score(yi, pi, average="weighted", labels=list(range(NC)), zero_division=0))
    try: row["macro_auc"] = roc_auc_score(yi, qi, multi_class="ovr", average="macro", labels=list(range(NC)))
    except ValueError: row["macro_auc"] = np.nan
    for k in range(NC):
        row[f"{C[k]}_recall"] = (pi[yi == k] == k).mean() if (yi == k).any() else np.nan
    boot.append(row)
boot = pd.DataFrame(boot); point = mv.full_metrics(y, hv, mp)
ci = pd.DataFrame({"estimate": {m: point[m] for m in boot.columns},
                   "ci95_low": boot.quantile(0.025), "ci95_high": boot.quantile(0.975)})
ci.to_csv(f"{OUT}/meff_ensemble_metrics_ci95.csv"); print((ci * 100).round(2).to_string())

## 3d. Confusion matrices (MEFF models and ensemble)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "savefig.dpi": 300})
def save(fig, name):
    for e in ("png", "pdf"): fig.savefig(f"{OUT}/{name}.{e}", bbox_inches="tight", facecolor="white")
    plt.show(); plt.close(fig)
BLUES = LinearSegmentedColormap.from_list("b", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
panels = {f"Expert {SEEDS.index(s)+1} (seed {s})": te[s].pred.values for s in SEEDS}; panels["MEFF (hard vote)"] = hv
fig, ax = plt.subplots(1, len(panels), figsize=(3.1 * len(panels), 3.5))
for a, (name, p) in zip(ax, panels.items()):
    cm_ = mv.confusion(y, p).astype(float); pct = 100 * cm_ / cm_.sum(1, keepdims=True)
    a.imshow(pct, cmap=BLUES, vmin=0, vmax=100)
    for i in range(NC):
        for j in range(NC):
            a.text(j, i, f"{pct[i, j]:.1f}%\n({int(cm_[i, j])})", ha="center", va="center", fontsize=7,
                   color="white" if pct[i, j] > 55 else "#1f1f1e")
    a.set_xticks(range(NC)); a.set_yticks(range(NC)); a.tick_params(length=0)
    a.set_xticklabels([k[:4] for k in C], fontsize=8); a.set_yticklabels([k[:4] for k in C] if a is ax[0] else [""] * NC, fontsize=8)
    a.set_xlabel("predicted"); a.set_title(name, fontsize=8.5)
    for sp_ in a.spines.values(): sp_.set_visible(False)
ax[0].set_ylabel("true")
fig.text(0.5, -0.06, "Rows sum to 100% (recall per class); counts in brackets. NONE, INFE(CTION), ISCH(AEMIA), BOTH.",
         ha="center", fontsize=8, color="#5f5e58")
fig.tight_layout(); save(fig, "fig_confusion")

## 3e. ROC and precision-recall curves (one-vs-rest)
The hard vote gives no score, so the curves use the ensemble's **mean softmax probability** (the same score as the macro-AUC).
Precision-recall curves are added because the classes are imbalanced (only 46 ISCHAEMIA test images).

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score
CC = ["#2a78d6", "#eb6834", "#1baf7a", "#c98500"]; LS = ["-", "--", "-.", ":"]
Y1 = np.eye(NC)[y]; grid = np.linspace(0, 1, 501)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.3))
tprs = []
for k in range(NC):
    f_, t_, _ = roc_curve(Y1[:, k], mp[:, k]); auc_k = roc_auc_score(Y1[:, k], mp[:, k]); tprs.append(np.interp(grid, f_, t_))
    ax[0].plot(f_, t_, color=CC[k], ls=LS[k], lw=1.8, label=f"{C[k]} (AUC {auc_k:.3f})")
    p_, r_, _ = precision_recall_curve(Y1[:, k], mp[:, k]); ap = average_precision_score(Y1[:, k], mp[:, k])
    ax[1].plot(r_, p_, color=CC[k], ls=LS[k], lw=1.8, label=f"{C[k]} (AP {ap:.3f}; prevalence {Y1[:, k].mean():.2f})")
macro_tpr = np.mean(tprs, 0); macro_tpr[0] = 0
ax[0].plot(grid, macro_tpr, color="#1f1f1e", lw=2.4, label=f"macro-average (AUC {roc_auc_score(y, mp, multi_class='ovr', average='macro'):.3f})")
ax[0].plot([0, 1], [0, 1], color="#c3c2b7", lw=1, ls=":")
for s, col, ls in zip(SEEDS, CC, LS):   # macro-average ROC of each single model vs the ensemble
    P = te[s][PC].values; t_all = [np.interp(grid, *roc_curve(Y1[:, k], P[:, k])[:2]) for k in range(NC)]
    m_ = np.mean(t_all, 0); m_[0] = 0
    ax[2].plot(grid, m_, color=col, ls=ls, lw=1.6, label=f"Expert {SEEDS.index(s)+1} (seed {s}) (AUC {roc_auc_score(y, P, multi_class='ovr', average='macro'):.3f})")
ax[2].plot(grid, macro_tpr, color="#1f1f1e", lw=2.4, label=f"MEFF, mean softmax (AUC {roc_auc_score(y, mp, multi_class='ovr', average='macro'):.3f})")
ax[2].plot([0, 1], [0, 1], color="#c3c2b7", lw=1, ls=":")
for a, t_, xl, yl in [(ax[0], "(a) ROC per class - MEFF", "false positive rate", "true positive rate"),
                      (ax[1], "(b) Precision-recall per class - MEFF", "recall", "precision"),
                      (ax[2], "(c) Macro-average ROC - single experts vs MEFF", "false positive rate", "true positive rate")]:
    a.set_title(t_, loc="left", fontsize=9.5); a.set_xlabel(xl); a.set_ylabel(yl); a.legend(fontsize=7.5, loc="lower right" if a is not ax[1] else "lower left")
    a.set_xlim(-0.01, 1.01); a.set_ylim(-0.01, 1.01); a.grid(color="#e4e3de", lw=0.6)
    for side in ("top", "right"): a.spines[side].set_visible(False)
fig.tight_layout(); save(fig, "fig_roc_pr")

## 4. Load the three trained models and check them against their saved Kaggle predictions

In [ ]:
import torch, torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
torch.set_grad_enabled(False)
TF = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
def rgb(rel): return Image.open(os.path.join(DATA, rel)).convert("RGB")

# handcrafted 42-D features, standardised with the TRAIN mean/SD exactly as in training
train = pd.read_csv(f"{SPLIT}/train.csv")
print(f"computing handcrafted features for {len(train)} training images (about 1-2 min) ...")
FTR = np.stack([mv.handcrafted_42(np.array(rgb(r))) for r in train.relpath])
MU, SD = FTR.mean(0), FTR.std(0); SD[SD < 1e-6] = 1.0
def hc(rel): return torch.tensor((mv.handcrafted_42(np.array(rgb(rel))) - MU) / SD, dtype=torch.float32)[None]

MODELS = {}
for s in SEEDS:
    m = mv.build_model("meff", pretrained=False).eval()
    ck = torch.load(CKPT[s], map_location="cpu", weights_only=False); m.load_state_dict(ck["model"]); MODELS[s] = m
    print(f"seed {s}: epoch {ck['epoch']}, val macro-F1 {ck['val_macro_f1']:.4f}")

# reproduce a sample of the saved test predictions (Kaggle used a GPU with mixed precision -> tiny differences)
chk = te[SEEDS[0]].groupby("true", group_keys=False).apply(lambda d: d.head(N_CHECK_PER_CLASS))
X = torch.stack([TF(rgb(r)) for r in chk.relpath]); H = torch.cat([hc(r) for r in chk.relpath])
for s in SEEDS:
    P = torch.softmax(MODELS[s](X, H), 1).numpy(); K = te[s].set_index("relpath").loc[chk.relpath, PC].values
    same = (P.argmax(1) == K.argmax(1)).mean(); diff = np.abs(P - K).max()
    print(f"seed {s}: same class {same:.0%} of {len(chk)} images, max |prob diff| {diff:.4f}")
    assert same >= 0.95 and diff < 0.05, f"seed {s}: checkpoint does not reproduce its saved predictions"
print("all checkpoints reproduce their saved predictions")

## 5. Which test images are shown (fixed rules, no hand-picking)
- **Correct examples:** for each class, the test image the ensemble classifies correctly with the *median* ensemble probability for that class.
- **Error examples:** for each class, the misclassified test image with the *median* ensemble probability for the wrongly predicted class.

In [ ]:
ens = te[SEEDS[0]][["relpath", "true"]].copy(); ens["pred"] = hv
for i, k in enumerate(C): ens[f"p_{k}"] = mp[:, i]
def median_pick(d, col):
    d = d.assign(_p=d.apply(lambda r: r[f"p_{C[int(r[col])]}"], axis=1)).sort_values(["_p", "relpath"]).reset_index(drop=True)
    return d.iloc[(len(d) - 1) // 2]
CORRECT, ERRORS = [], []
for i, k in enumerate(C):
    ok = ens[(ens.true == i) & (ens.pred == i)]; bad = ens[(ens.true == i) & (ens.pred != i)]
    if len(ok): CORRECT.append(median_pick(ok, "true"))
    if len(bad): ERRORS.append(median_pick(bad, "pred"))
print("correct:", [r.relpath for r in CORRECT]); print("errors: ", [(r.relpath, "->", C[int(r.pred)]) for r in ERRORS])
with open(f"{OUT}/figure_notes.txt", "w") as f:
    f.write("correct examples (median ensemble probability, correct): " + ", ".join(r.relpath for r in CORRECT) + "\n")
    f.write("error examples (median probability of the wrong class): " + ", ".join(f"{r.relpath}->{C[int(r.pred)]}" for r in ERRORS) + "\n")

## 6. Feature maps (one model)
Columns: the test image; the strongest channel after dense blocks 1, 2 and 3; and the mean of all 1,664 channels of the
7 × 7 map after CCDGS + Triplet Attention + SE (the map that is pooled), overlaid on the image.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "savefig.dpi": 300})
STAGES = [("block1", "Dense block 1\n256 × 56 × 56"), ("block2", "Dense block 2\n512 × 28 × 28"),
          ("block3", "Dense block 3\n1280 × 14 × 14"), ("final", "CCDGS + TA + SE\n1664 × 7 × 7")]

def activations(model, x):
    acts = {}
    hooks = [getattr(model.features, f"denseblock{i}").register_forward_hook(
        lambda m, i_, o, k=f"block{i}": acts.__setitem__(k, o.detach())) for i in (1, 2, 3)]
    f = F.relu(model.features(x)); acts["final"] = model.se(model.ta(model.ccdgs(f))).detach()
    for h in hooks: h.remove()
    return {k: v[0].float().numpy() for k, v in acts.items()}

def norm01(a, robust=False):
    if robust and min(a.shape) > 4:   # zero-padding makes the 1-cell border extreme: scale on the interior
        lo, hi = np.percentile(a[1:-1, 1:-1], [1, 99])
        return np.clip((a - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros_like(a)
    a = a - a.min(); return a / a.max() if a.max() > 0 else a

def overlay(ax, img, m, alpha=0.55):
    ax.imshow(img); ax.imshow(np.array(Image.fromarray((255 * norm01(m)).astype(np.uint8)).resize((224, 224), Image.BILINEAR)),
                              cmap="inferno", alpha=alpha)

def save(fig, name):
    for e in ("png", "pdf"): fig.savefig(f"{OUT}/{name}.{e}", bbox_inches="tight", facecolor="white")
    plt.show(); plt.close(fig)

model = MODELS[SEED_FOR_FEATUREMAPS]
res_fm = [(r.relpath, np.array(rgb(r.relpath).resize((224, 224))), activations(model, TF(rgb(r.relpath))[None])) for r in CORRECT]
fig, ax = plt.subplots(len(res_fm), 5, figsize=(10.5, 2.3 * len(res_fm)))
for r, (rel, img, A) in enumerate(res_fm):
    ax[r, 0].imshow(img); ax[r, 0].set_ylabel(f"({'abcd'[r]}) {rel.split('/')[0]}")
    for c_, (k, lab) in enumerate(STAGES, 1):
        if k == "final":
            overlay(ax[r, c_], img, A[k].mean(0))
        else:
            top = int(np.argmax(A[k].mean((1, 2)))); ax[r, c_].imshow(norm01(A[k][top], True), cmap="inferno")
            ax[r, c_].set_xlabel(f"channel {top}", fontsize=7.5)
        if r == 0: ax[r, c_].set_title(lab, fontsize=8.5)
    if r == 0: ax[r, 0].set_title("Test image", fontsize=8.5)
    for x in ax[r]: x.set_xticks([]); x.set_yticks([])
fig.text(0.5, -0.035, f"Expert {SEEDS.index(SEED_FOR_FEATUREMAPS)+1} (seed {SEED_FOR_FEATUREMAPS}). Blocks 1-3: the channel with the highest mean activation (scaled on the interior, 1st-99th percentile). "
         "Last column: mean of all 1,664 channels, upsampled and overlaid.", ha="center", fontsize=7.5, color="#5f5e58")
fig.tight_layout(); save(fig, "fig_featuremaps_overview")

for rel, img, A in res_fm:   # per image: the 6 strongest channels at each stage
    fig, ax = plt.subplots(len(STAGES), 7, figsize=(13, 8))
    for r, (k, lab) in enumerate(STAGES):
        ax[r, 0].imshow(img); ax[r, 0].set_ylabel(lab, fontsize=8.5)
        for c_, ch in enumerate(np.argsort(A[k].mean((1, 2)))[::-1][:6], 1):
            ax[r, c_].imshow(norm01(A[k][ch], k != "final"), cmap="inferno"); ax[r, c_].set_title(f"channel {ch}", fontsize=7.5)
        for x in ax[r]: x.set_xticks([]); x.set_yticks([])
    fig.suptitle(f"{rel}: the 6 channels with the highest mean activation at each stage (Expert {SEEDS.index(SEED_FOR_FEATUREMAPS)+1}, seed {SEED_FOR_FEATUREMAPS})", fontsize=10)
    fig.tight_layout(); save(fig, f"fig_featuremaps_{rel.split('/')[0]}")

## 7. Grad-CAM for each model and for the ensemble
- Target layer: the 1664 × 7 × 7 map after CCDGS + Triplet Attention + SE (the last spatial map before pooling).
- Target class: the **ensemble's** hard-vote prediction, so all three models explain the same decision.
- Grad-CAM = ReLU(Σ_c w_c · A_c), with w_c = spatial mean of ∂(logit of the target class)/∂A_c.
- **Ensemble Grad-CAM** = mean of the three models' maps, each scaled to 0–1 first (equal weight per model).
- The 42-D handcrafted features also feed the classifier but have no spatial layout, so Grad-CAM shows only the image branch.

In [ ]:
def gradcam(model, rel, target):
    x = TF(rgb(rel))[None]; h = hc(rel)
    with torch.enable_grad():
        a = model.se(model.ta(model.ccdgs(F.relu(model.features(x)))))
        a.retain_grad()
        logit = model.head(torch.cat([a.mean((2, 3)), h], 1))
        model.zero_grad(); logit[0, target].backward()
    w = a.grad.mean((2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1))[0].detach().numpy()
    return cam, float(torch.softmax(logit.detach(), 1)[0, target])

def gradcam_figure(picks, name, title):
    if not picks: print("no images for", name); return
    fig, ax = plt.subplots(len(picks), 2 + len(SEEDS), figsize=(2.3 * (2 + len(SEEDS)), 2.45 * len(picks)))
    ax = np.atleast_2d(ax)
    for r, row in enumerate(picks):
        rel, t, p = row.relpath, int(row.true), int(row.pred)
        img = np.array(rgb(rel).resize((224, 224))); cams = []
        ax[r, 0].imshow(img); ax[r, 0].set_ylabel(f"({'abcd'[r]}) true {C[t]}", fontsize=8.5)
        for j, s in enumerate(SEEDS, 1):
            cam, prob = gradcam(MODELS[s], rel, p); cams.append(norm01(cam))
            overlay(ax[r, j], img, cam); ax[r, j].set_xlabel(f"p({C[p]}) = {prob:.2f}", fontsize=7.5)
        overlay(ax[r, -1], img, np.mean(cams, 0)); ax[r, -1].set_xlabel(f"MEFF: {C[p]} (p = {row[f'p_{C[p]}']:.2f})", fontsize=7.5)
        for x in ax[r]: x.set_xticks([]); x.set_yticks([])
    for j, t_ in enumerate(["Test image"] + [f"Grad-CAM\nExpert {SEEDS.index(s)+1} (seed {s})" for s in SEEDS] + ["Grad-CAM\nMEFF (ensemble)"]):
        ax[0, j].set_title(t_, fontsize=8.5)
    fig.suptitle(title, fontsize=10); fig.tight_layout(); save(fig, name)

gradcam_figure(CORRECT, "fig_gradcam_correct", "Grad-CAM for the ensemble's prediction - correctly classified test images")
gradcam_figure(ERRORS, "fig_gradcam_errors", "Grad-CAM for the ensemble's (wrong) prediction - misclassified test images")

## 8. Download

In [ ]:
out = f"{W}/MEFF_v3_analysis_outputs.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for f in fs: z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print("Download from the Output tab:", os.path.basename(out)); print(sorted(os.listdir(OUT)))